In [3]:

# GAN Implementation

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# =========================
# 1. Hyperparameters
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size   = 256
z_dim        = 2      # noise dimension
x_dim        = 2      # data is 2D points on unit circle
hidden_dim   = 128
num_epochs   = 50
steps_per_epoch = 200   # how many batches per epoch
lr           = 2e-4

# =========================
# 2. Real data sampler: X = Z / ||Z||
# =========================
def sample_real(batch_size):
    z = torch.randn(batch_size, x_dim, device=device)
    norms = torch.norm(z, dim=1, keepdim=True) + 1e-8
    x = z / norms
    return x

# =========================
# 3. Generator and Discriminator
# =========================
class Generator(nn.Module):
    def __init__(self, z_dim, x_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, x_dim),
            nn.Tanh()          # outputs roughly in [-1, 1]^2 (good for unit circle)
        )

    def forward(self, z):
        return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, x_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(x_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1)  # we'll use BCEWithLogitsLoss
        )

    def forward(self, x):
        return self.net(x)

G = Generator(z_dim, x_dim, hidden_dim).to(device)
D = Discriminator(x_dim, hidden_dim).to(device)

criterion = nn.BCEWithLogitsLoss()
opt_G = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))

# =========================
# 4. Helper: plot generated samples after each epoch
# =========================
def plot_generated(epoch, num_points=1000):
    G.eval()
    with torch.no_grad():
        z = torch.randn(num_points, z_dim, device=device)
        x_fake = G(z).cpu().numpy()

    # also sample a small real batch for reference
    x_real = sample_real(num_points).cpu().numpy()

    plt.figure(figsize=(4, 4))
    # real data (light)
    plt.scatter(x_real[:, 0], x_real[:, 1], s=5, alpha=0.2, label="Real")
    # generated data
    plt.scatter(x_fake[:, 0], x_fake[:, 1], s=5, alpha=0.8, label="Generated")
    plt.gca().set_aspect("equal", "box")
    plt.title(f"Generated samples after epoch {epoch+1}")
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()          # or plt.savefig(f"epoch_{epoch+1:03d}.png"); plt.close()

    G.train()

# --- add these imports at the top ---
import os
from pathlib import Path
import imageio.v2 as imageio   # pip install imageio
# (optional) from PIL import Image

# --- add this near hyperparameters / config ---
frames_dir = Path("gan_frames")
frames_dir.mkdir(parents=True, exist_ok=True)
gif_path = Path("gan_training.gif")

# =========================
# 4. Helper: save generated samples after each epoch (frame)
# =========================
def save_generated_frame(epoch, num_points=1000):
    G.eval()
    with torch.no_grad():
        z = torch.randn(num_points, z_dim, device=device)
        x_fake = G(z).cpu().numpy()

    x_real = sample_real(num_points).cpu().numpy()

    plt.figure(figsize=(4, 4))
    plt.scatter(x_real[:, 0], x_real[:, 1], s=5, alpha=0.2, label="Real")
    plt.scatter(x_fake[:, 0], x_fake[:, 1], s=5, alpha=0.8, label="Generated")

    ax = plt.gca()
    ax.set_aspect("equal", "box")

    # Fix axis limits so the GIF doesn't "jump" frame-to-frame
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)

    plt.title(f"Generated samples after epoch {epoch+1}")
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    frame_file = frames_dir / f"epoch_{epoch+1:03d}.png"
    plt.savefig(frame_file, dpi=150)
    plt.close()
    G.train()

# =========================
# 5. Training loop
# =========================
for epoch in range(num_epochs):

    for step in range(steps_per_epoch):
        # ---- Train Discriminator ----
        x_real = sample_real(batch_size)
        y_real = torch.ones(batch_size, 1, device=device)

        z = torch.randn(batch_size, z_dim, device=device)
        x_fake = G(z).detach()
        y_fake = torch.zeros(batch_size, 1, device=device)

        D_real = D(x_real)
        D_fake = D(x_fake)
        loss_D = criterion(D_real, y_real) + criterion(D_fake, y_fake)

        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        # ---- Train Generator ----
        z = torch.randn(batch_size, z_dim, device=device)
        x_fake = G(z)
        D_fake = D(x_fake)
        loss_G = criterion(D_fake, y_real)

        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

    print(f"Epoch {epoch+1}/{num_epochs} | loss_D={loss_D.item():.3f} | loss_G={loss_G.item():.3f}")
    save_generated_frame(epoch)   # <-- save frame instead of plt.show()

# =========================
# 6. Make GIF from frames
# =========================
frame_files = sorted(frames_dir.glob("epoch_*.png"))
images = [imageio.imread(f) for f in frame_files]
imageio.mimsave(gif_path, images, duration=0.25, loop=0)  # duration in seconds per frame

print(f"Saved GIF to: {gif_path.resolve()}")

Epoch 1/50 | loss_D=1.259 | loss_G=0.910
Epoch 2/50 | loss_D=1.460 | loss_G=0.737
Epoch 3/50 | loss_D=1.391 | loss_G=0.697
Epoch 4/50 | loss_D=1.295 | loss_G=0.864
Epoch 5/50 | loss_D=1.066 | loss_G=0.975
Epoch 6/50 | loss_D=1.408 | loss_G=0.639
Epoch 7/50 | loss_D=1.425 | loss_G=0.708
Epoch 8/50 | loss_D=1.228 | loss_G=0.883
Epoch 9/50 | loss_D=1.345 | loss_G=0.702
Epoch 10/50 | loss_D=1.201 | loss_G=0.878
Epoch 11/50 | loss_D=0.986 | loss_G=1.119
Epoch 12/50 | loss_D=1.189 | loss_G=1.017
Epoch 13/50 | loss_D=0.956 | loss_G=1.022
Epoch 14/50 | loss_D=1.005 | loss_G=1.180
Epoch 15/50 | loss_D=1.350 | loss_G=0.762
Epoch 16/50 | loss_D=0.812 | loss_G=1.216
Epoch 17/50 | loss_D=1.114 | loss_G=1.146
Epoch 18/50 | loss_D=0.682 | loss_G=1.406
Epoch 19/50 | loss_D=1.167 | loss_G=1.001
Epoch 20/50 | loss_D=0.893 | loss_G=1.188
Epoch 21/50 | loss_D=0.966 | loss_G=1.100
Epoch 22/50 | loss_D=1.038 | loss_G=1.029
Epoch 23/50 | loss_D=0.926 | loss_G=1.130
Epoch 24/50 | loss_D=1.197 | loss_G=0.866
E

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

import imageio.v2 as imageio   # pip install imageio
from pathlib import Path

# =========================
# 1. Hyperparameters
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size      = 256
z_dim           = 2
x_dim           = 2
hidden_dim      = 128
num_epochs      = 50
steps_per_epoch = 200

lr              = 5e-5
lambda_gp       = 5.0
n_critic        = 10

torch.manual_seed(0)

# ---- GIF config ----
frames_dir = Path("wgan_gp_frames")
frames_dir.mkdir(parents=True, exist_ok=True)
gif_path = Path("wgan_gp_training.gif")

# =========================
# 2. Real data sampler: X = Z / ||Z||
# =========================
def sample_real(batch_size):
    z = torch.randn(batch_size, x_dim, device=device)
    norms = torch.norm(z, dim=1, keepdim=True) + 1e-8
    x = z / norms
    return x

# =========================
# 3. Generator and Critic
# =========================
class Generator(nn.Module):
    def __init__(self, z_dim, x_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, x_dim),
            nn.Tanh()
        )
    def forward(self, z):
        return self.net(z)

class Critic(nn.Module):
    def __init__(self, x_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(x_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, x):
        return self.net(x)

G = Generator(z_dim, x_dim, hidden_dim).to(device)
C = Critic(x_dim, hidden_dim).to(device)

opt_G = optim.Adam(G.parameters(), lr=lr, betas=(0.0, 0.9))
opt_C = optim.Adam(C.parameters(), lr=lr, betas=(0.0, 0.9))

# =========================
# 4. Gradient penalty
# =========================
def gradient_penalty(critic, real, fake):
    batch_size = real.size(0)
    eps = torch.rand(batch_size, 1, device=device).expand_as(real)

    interpolates = eps * real + (1 - eps) * fake
    interpolates.requires_grad_(True)

    critic_interpolates = critic(interpolates)

    grads = torch.autograd.grad(
        outputs=critic_interpolates,
        inputs=interpolates,
        grad_outputs=torch.ones_like(critic_interpolates),
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]

    grads = grads.view(batch_size, -1)
    grad_norm = grads.norm(2, dim=1)
    gp = ((grad_norm - 1.0) ** 2).mean()
    return gp

# =========================
# 5. Save generated samples frame (instead of plt.show)
# =========================
def save_generated_frame(epoch, c_loss=None, g_loss=None, num_points=2000):
    G.eval()
    with torch.no_grad():
        z = torch.randn(num_points, z_dim, device=device)
        x_fake = G(z).cpu().numpy()
    x_real = sample_real(num_points).cpu().numpy()

    plt.figure(figsize=(4, 4))
    plt.scatter(x_real[:, 0], x_real[:, 1], s=5, alpha=0.2, label="Real")
    plt.scatter(x_fake[:, 0], x_fake[:, 1], s=5, alpha=0.8, label="Generated")

    ax = plt.gca()
    ax.set_aspect("equal", "box")

    # Fix limits so GIF doesn't "jump"
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)

    title = f"WGAN-GP samples after epoch {epoch+1}"
    if (c_loss is not None) and (g_loss is not None):
        title += f"\nC_loss={c_loss:.4f} | G_loss={g_loss:.4f}"

    plt.title(title)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    frame_file = frames_dir / f"epoch_{epoch+1:03d}.png"
    plt.savefig(frame_file, dpi=150)
    plt.close()
    G.train()

# =========================
# 6. Training loop
# =========================
for epoch in range(num_epochs):
    epoch_c_loss = 0.0
    epoch_g_loss = 0.0

    for step in range(steps_per_epoch):

        # -------------------
        # Train Critic
        # -------------------
        for _ in range(n_critic):
            x_real = sample_real(batch_size)
            z = torch.randn(batch_size, z_dim, device=device)
            x_fake = G(z).detach()

            C_real = C(x_real)
            C_fake = C(x_fake)

            loss_C = C_fake.mean() - C_real.mean()
            gp = gradient_penalty(C, x_real, x_fake)
            loss_C_total = loss_C + lambda_gp * gp

            opt_C.zero_grad()
            loss_C_total.backward()
            opt_C.step()

        # -------------------
        # Train Generator
        # -------------------
        z = torch.randn(batch_size, z_dim, device=device)
        x_fake = G(z)
        C_fake = C(x_fake)
        loss_G = -C_fake.mean()

        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

        epoch_c_loss += loss_C.item()
        epoch_g_loss += loss_G.item()

    epoch_c_loss /= steps_per_epoch
    epoch_g_loss /= steps_per_epoch

    print(f"Epoch {epoch+1}/{num_epochs} | C_loss={epoch_c_loss:.4f} | G_loss={epoch_g_loss:.4f}")

    # save epoch frame
    save_generated_frame(epoch, c_loss=epoch_c_loss, g_loss=epoch_g_loss)

# =========================
# 7. Make GIF from frames
# =========================
frame_files = sorted(frames_dir.glob("epoch_*.png"))
images = [imageio.imread(f) for f in frame_files]
imageio.mimsave(gif_path, images, duration=0.25, loop=0)

print(f"Saved GIF to: {gif_path.resolve()}")

Epoch 1/50 | C_loss=-0.5425 | G_loss=0.1807
Epoch 2/50 | C_loss=-0.2636 | G_loss=-0.5293
Epoch 3/50 | C_loss=-0.0452 | G_loss=-0.9839
Epoch 4/50 | C_loss=-0.1062 | G_loss=-0.9288
Epoch 5/50 | C_loss=-0.0709 | G_loss=-1.3035
Epoch 6/50 | C_loss=-0.0516 | G_loss=-1.3446
Epoch 7/50 | C_loss=-0.0356 | G_loss=-1.4048
Epoch 8/50 | C_loss=-0.0114 | G_loss=-1.6826
Epoch 9/50 | C_loss=-0.0138 | G_loss=-1.7618
Epoch 10/50 | C_loss=0.0047 | G_loss=-2.0429
Epoch 11/50 | C_loss=0.0067 | G_loss=-2.2603
Epoch 12/50 | C_loss=-0.0014 | G_loss=-2.4022
Epoch 13/50 | C_loss=-0.0088 | G_loss=-2.2127
Epoch 14/50 | C_loss=-0.0087 | G_loss=-2.3224
Epoch 15/50 | C_loss=-0.0113 | G_loss=-2.5375
Epoch 16/50 | C_loss=-0.0110 | G_loss=-2.8165
Epoch 17/50 | C_loss=-0.0276 | G_loss=-2.9962
Epoch 18/50 | C_loss=-0.0305 | G_loss=-2.9823
Epoch 19/50 | C_loss=-0.0561 | G_loss=-3.1413
Epoch 20/50 | C_loss=-0.0556 | G_loss=-3.0484
Epoch 21/50 | C_loss=-0.0474 | G_loss=-3.0506
Epoch 22/50 | C_loss=-0.0610 | G_loss=-2.8849


In [5]:
# VAE Implementation + GIF

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

import imageio.v2 as imageio   # pip install imageio
from pathlib import Path

# =========================
# 1. Hyperparameters
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size      = 256
x_dim           = 2
hidden_dim      = 128
latent_dim      = 2
num_epochs      = 50
steps_per_epoch = 200
lr              = 1e-3
beta_kl         = 1.0

# ---- GIF config ----
frames_dir = Path("vae_frames")
frames_dir.mkdir(parents=True, exist_ok=True)
gif_path = Path("vae_training.gif")

# =========================
# 2. Real data sampler: X = Z / ||Z||
# =========================
def sample_real(batch_size):
    z = torch.randn(batch_size, x_dim, device=device)
    norms = torch.norm(z, dim=1, keepdim=True) + 1e-8
    x = z / norms
    return x

# =========================
# 3. VAE model
# =========================
class VAE(nn.Module):
    def __init__(self, x_dim, hidden_dim, latent_dim):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(x_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.mu_head     = nn.Linear(hidden_dim, latent_dim)
        self.logvar_head = nn.Linear(hidden_dim, latent_dim)

        self.dec = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, x_dim),
            nn.Tanh()
        )

    def encode(self, x):
        h = self.enc(x)
        mu     = self.mu_head(h)
        logvar = self.logvar_head(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.dec(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z)
        return x_recon, mu, logvar

vae = VAE(x_dim, hidden_dim, latent_dim).to(device)
optimizer = optim.Adam(vae.parameters(), lr=lr)

# =========================
# 4. Loss function
# =========================
def vae_loss(x, x_recon, mu, logvar, beta=1.0):
    recon = nn.functional.mse_loss(x_recon, x, reduction="sum")
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return (recon + beta * kl) / x.size(0), recon / x.size(0), kl / x.size(0)

# =========================
# 5. Save generated samples frame (instead of plt.show)
# =========================
def save_generated_frame(epoch, loss=None, recon=None, kl=None, num_points=1000):
    vae.eval()
    with torch.no_grad():
        z = torch.randn(num_points, latent_dim, device=device)
        x_fake = vae.decode(z).cpu().numpy()

    x_real = sample_real(num_points).cpu().numpy()

    plt.figure(figsize=(4, 4))
    plt.scatter(x_real[:, 0], x_real[:, 1], s=5, alpha=0.2, label="Real")
    plt.scatter(x_fake[:, 0], x_fake[:, 1], s=5, alpha=0.8, label="Generated")

    ax = plt.gca()
    ax.set_aspect("equal", "box")

    # Fix axis limits to avoid jitter between frames
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)

    title = f"VAE samples after epoch {epoch+1}"
    if (loss is not None) and (recon is not None) and (kl is not None):
        title += f"\nloss={loss:.4f} | recon={recon:.4f} | KL={kl:.4f}"

    plt.title(title)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    frame_file = frames_dir / f"epoch_{epoch+1:03d}.png"
    plt.savefig(frame_file, dpi=150)
    plt.close()
    vae.train()

# =========================
# 6. Training loop
# =========================
for epoch in range(num_epochs):
    epoch_loss = 0.0
    epoch_recon = 0.0
    epoch_kl = 0.0

    for step in range(steps_per_epoch):
        x = sample_real(batch_size)

        x_recon, mu, logvar = vae(x)
        loss, recon, kl = vae_loss(x, x_recon, mu, logvar, beta=beta_kl)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        epoch_recon += recon.item()
        epoch_kl += kl.item()

    epoch_loss /= steps_per_epoch
    epoch_recon /= steps_per_epoch
    epoch_kl /= steps_per_epoch

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"loss={epoch_loss:.4f} | recon={epoch_recon:.4f} | KL={epoch_kl:.4f}"
    )

    # save epoch frame
    save_generated_frame(epoch, loss=epoch_loss, recon=epoch_recon, kl=epoch_kl)

# =========================
# 7. Make GIF from frames
# =========================
frame_files = sorted(frames_dir.glob("epoch_*.png"))
images = [imageio.imread(f) for f in frame_files]
imageio.mimsave(gif_path, images, duration=0.25, loop=0)

print(f"Saved GIF to: {gif_path.resolve()}")

Epoch 1/50 | loss=1.0023 | recon=0.9780 | KL=0.0243
Epoch 2/50 | loss=1.0009 | recon=0.9823 | KL=0.0186
Epoch 3/50 | loss=1.0011 | recon=0.9908 | KL=0.0103
Epoch 4/50 | loss=1.0004 | recon=0.9944 | KL=0.0060
Epoch 5/50 | loss=1.0012 | recon=0.9974 | KL=0.0039
Epoch 6/50 | loss=1.0002 | recon=0.9995 | KL=0.0007
Epoch 7/50 | loss=1.0004 | recon=0.9991 | KL=0.0013
Epoch 8/50 | loss=1.0002 | recon=0.9985 | KL=0.0017
Epoch 9/50 | loss=1.0004 | recon=0.9989 | KL=0.0014
Epoch 10/50 | loss=1.0005 | recon=0.9997 | KL=0.0007
Epoch 11/50 | loss=1.0003 | recon=0.9999 | KL=0.0003
Epoch 12/50 | loss=1.0001 | recon=1.0000 | KL=0.0001
Epoch 13/50 | loss=1.0001 | recon=1.0001 | KL=0.0001
Epoch 14/50 | loss=1.0001 | recon=0.9998 | KL=0.0004
Epoch 15/50 | loss=1.0003 | recon=1.0001 | KL=0.0002
Epoch 16/50 | loss=1.0001 | recon=1.0000 | KL=0.0001
Epoch 17/50 | loss=1.0000 | recon=0.9997 | KL=0.0003
Epoch 18/50 | loss=1.0001 | recon=0.9999 | KL=0.0003
Epoch 19/50 | loss=1.0000 | recon=0.9996 | KL=0.0004
Ep

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

import imageio.v2 as imageio   # pip install imageio
from pathlib import Path

# =========================
# 1. Hyperparameters
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size      = 256
x_dim           = 2
hidden_dim      = 256
latent_dim      = 2
num_epochs      = 80
steps_per_epoch = 300
lr              = 1e-3

beta_kl         = 0.01
lambda_radial   = 1.0

# ---- GIF config ----
frames_dir = Path("vae_radial_frames")
frames_dir.mkdir(parents=True, exist_ok=True)
gif_path = Path("vae_radial_training.gif")

# =========================
# 2. Real data sampler: X = Z / ||Z||
# =========================
def sample_real(batch_size):
    z = torch.randn(batch_size, x_dim, device=device)
    norms = torch.norm(z, dim=1, keepdim=True) + 1e-8
    x = z / norms
    return x

# =========================
# 3. VAE model
# =========================
class VAE(nn.Module):
    def __init__(self, x_dim, hidden_dim, latent_dim):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(x_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.mu_head     = nn.Linear(hidden_dim, latent_dim)
        self.logvar_head = nn.Linear(hidden_dim, latent_dim)

        self.dec = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, x_dim),
            nn.Tanh()
        )

    def encode(self, x):
        h = self.enc(x)
        mu     = self.mu_head(h)
        logvar = self.logvar_head(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.dec(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z)
        return x_recon, mu, logvar

vae = VAE(x_dim, hidden_dim, latent_dim).to(device)
optimizer = optim.Adam(vae.parameters(), lr=lr)

# =========================
# 4. Loss function with radial penalty
# =========================
def vae_loss(x, x_recon, mu, logvar, beta=1.0, lambda_radial=1.0):
    recon = nn.functional.mse_loss(x_recon, x, reduction="sum")

    r = torch.norm(x_recon, dim=1)
    radial = ((r - 1.0) ** 2).sum()

    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    total = (recon + lambda_radial * radial + beta * kl) / x.size(0)
    return total, recon / x.size(0), radial / x.size(0), kl / x.size(0)

# =========================
# 5. Save epoch frame (instead of plt.show)
# =========================
def save_generated_frame(epoch, loss=None, recon=None, radial=None, kl=None, num_points=2000):
    vae.eval()
    with torch.no_grad():
        z = torch.randn(num_points, latent_dim, device=device)
        x_fake = vae.decode(z).cpu().numpy()
    x_real = sample_real(num_points).cpu().numpy()

    plt.figure(figsize=(4, 4))
    plt.scatter(x_real[:, 0], x_real[:, 1], s=5, alpha=0.2, label="Real")
    plt.scatter(x_fake[:, 0], x_fake[:, 1], s=5, alpha=0.8, label="Generated")

    ax = plt.gca()
    ax.set_aspect("equal", "box")

    # Fix axis limits to avoid jitter in GIF
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)

    title = f"VAE (radial penalty) after epoch {epoch+1}"
    if all(v is not None for v in [loss, recon, radial, kl]):
        title += f"\nloss={loss:.4f} | recon={recon:.4f} | radial={radial:.4f} | KL={kl:.4f}"

    plt.title(title)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    frame_file = frames_dir / f"epoch_{epoch+1:03d}.png"
    plt.savefig(frame_file, dpi=150)
    plt.close()
    vae.train()

# =========================
# 6. Training loop
# =========================
for epoch in range(num_epochs):
    epoch_loss = epoch_recon = epoch_radial = epoch_kl = 0.0

    for step in range(steps_per_epoch):
        x = sample_real(batch_size)

        x_recon, mu, logvar = vae(x)
        loss, recon, radial, kl = vae_loss(
            x, x_recon, mu, logvar,
            beta=beta_kl,
            lambda_radial=lambda_radial
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss   += loss.item()
        epoch_recon  += recon.item()
        epoch_radial += radial.item()
        epoch_kl     += kl.item()

    epoch_loss   /= steps_per_epoch
    epoch_recon  /= steps_per_epoch
    epoch_radial /= steps_per_epoch
    epoch_kl     /= steps_per_epoch

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"loss={epoch_loss:.4f} | recon={epoch_recon:.4f} | "
        f"radial={epoch_radial:.4f} | KL={epoch_kl:.4f}"
    )

    save_generated_frame(epoch, loss=epoch_loss, recon=epoch_recon, radial=epoch_radial, kl=epoch_kl)

# =========================
# 7. Make GIF from frames
# =========================
frame_files = sorted(frames_dir.glob("epoch_*.png"))
images = [imageio.imread(f) for f in frame_files]
imageio.mimsave(gif_path, images, duration=0.2, loop=0)

print(f"Saved GIF to: {gif_path.resolve()}")

Epoch 1/80 | loss=0.1185 | recon=0.0653 | radial=0.0128 | KL=4.0427
Epoch 2/80 | loss=0.0481 | recon=0.0095 | radial=0.0006 | KL=3.8115
Epoch 3/80 | loss=0.0451 | recon=0.0078 | radial=0.0004 | KL=3.6968
Epoch 4/80 | loss=0.0445 | recon=0.0074 | radial=0.0003 | KL=3.6811
Epoch 5/80 | loss=0.0443 | recon=0.0074 | radial=0.0003 | KL=3.6581
Epoch 6/80 | loss=0.0441 | recon=0.0073 | radial=0.0002 | KL=3.6556
Epoch 7/80 | loss=0.0439 | recon=0.0072 | radial=0.0002 | KL=3.6485
Epoch 8/80 | loss=0.0440 | recon=0.0073 | radial=0.0002 | KL=3.6470
Epoch 9/80 | loss=0.0437 | recon=0.0071 | radial=0.0002 | KL=3.6386
Epoch 10/80 | loss=0.0440 | recon=0.0073 | radial=0.0002 | KL=3.6452
Epoch 11/80 | loss=0.0436 | recon=0.0071 | radial=0.0002 | KL=3.6338
Epoch 12/80 | loss=0.0436 | recon=0.0072 | radial=0.0002 | KL=3.6242
Epoch 13/80 | loss=0.0434 | recon=0.0071 | radial=0.0002 | KL=3.6173
Epoch 14/80 | loss=0.0433 | recon=0.0068 | radial=0.0002 | KL=3.6310
Epoch 15/80 | loss=0.0434 | recon=0.0072 | 

In [7]:
import math
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import matplotlib.pyplot as plt
import imageio.v2 as imageio   # pip install imageio

# =========================================================
# 0. Config
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)

# Data: 2D unit circle
x_dim = 2

# Training
batch_size = 512
num_epochs = 50
steps_per_epoch = 300
lr = 2e-4

# Diffusion steps
T = 200  # number of diffusion timesteps

# GIF settings
out_dir = Path("ddpm_gifs")
out_dir.mkdir(exist_ok=True, parents=True)

frames_conv_dir = out_dir / "frames_convergence"
frames_path_dir = out_dir / "frames_inference_path"
frames_conv_dir.mkdir(exist_ok=True, parents=True)
frames_path_dir.mkdir(exist_ok=True, parents=True)

gif_convergence = out_dir / "ddpm_training_convergence.gif"
gif_inference_path = out_dir / "ddpm_inference_path.gif"

# =========================================================
# 1. Real data sampler: X = Z / ||Z||
# =========================================================
def sample_real(n):
    z = torch.randn(n, x_dim, device=device)
    norms = torch.norm(z, dim=1, keepdim=True) + 1e-8
    return z / norms

# =========================================================
# 2. Time embedding (sin/cos) + epsilon network (MLP)
# =========================================================
class TimeEmbedding(nn.Module):
    """Classic sinusoidal embedding for integer t in [1..T]."""
    def __init__(self, embed_dim: int):
        super().__init__()
        self.embed_dim = embed_dim

    def forward(self, t: torch.Tensor):
        # t: (B,) integer, on device
        half = self.embed_dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(0, half, device=t.device).float() / half
        )  # (half,)
        # (B, half)
        angles = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        emb = torch.cat([torch.sin(angles), torch.cos(angles)], dim=1)  # (B, 2*half)
        if self.embed_dim % 2 == 1:
            emb = torch.cat([emb, torch.zeros(t.size(0), 1, device=t.device)], dim=1)
        return emb

class EpsNet(nn.Module):
    """Predict epsilon given x_t and t."""
    def __init__(self, x_dim=2, hidden=256, t_dim=64):
        super().__init__()
        self.t_embed = TimeEmbedding(t_dim)
        self.net = nn.Sequential(
            nn.Linear(x_dim + t_dim, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, x_dim),
        )

    def forward(self, x_t, t):
        # x_t: (B,2), t: (B,)
        te = self.t_embed(t)                # (B, t_dim)
        h = torch.cat([x_t, te], dim=1)     # (B, 2 + t_dim)
        return self.net(h)                 # (B,2)

eps_model = EpsNet(x_dim=2, hidden=256, t_dim=64).to(device)
opt = optim.Adam(eps_model.parameters(), lr=lr)

# =========================================================
# 3. DDPM schedules
# =========================================================
def make_linear_beta_schedule(T, beta_start=1e-4, beta_end=2e-2):
    return torch.linspace(beta_start, beta_end, T, device=device)

betas = make_linear_beta_schedule(T)               # (T,)
alphas = 1.0 - betas                               # (T,)
alpha_bars = torch.cumprod(alphas, dim=0)          # (T,)

# Convenience terms (all (T,))
sqrt_alpha_bars = torch.sqrt(alpha_bars)
sqrt_one_minus_alpha_bars = torch.sqrt(1.0 - alpha_bars)
sqrt_recip_alphas = torch.sqrt(1.0 / alphas)

# For posterior variance (classic choice: beta_t)
# (works fine for this 2D toy demo)
sigmas = torch.sqrt(betas)

# =========================================================
# 4. Forward diffusion: q(x_t | x0)
#    x_t = sqrt(a_bar_t)*x0 + sqrt(1-a_bar_t)*eps
# =========================================================
def q_sample(x0, t, eps=None):
    """
    x0: (B,2)
    t:  (B,) integer in [1..T]
    """
    if eps is None:
        eps = torch.randn_like(x0)
    # gather correct coefficients for each sample in batch
    # note: our arrays are 0-indexed, so use (t-1)
    c1 = sqrt_alpha_bars[t - 1].unsqueeze(1)              # (B,1)
    c2 = sqrt_one_minus_alpha_bars[t - 1].unsqueeze(1)    # (B,1)
    return c1 * x0 + c2 * eps, eps

# =========================================================
# 5. Reverse diffusion: p_theta(x_{t-1} | x_t)
# =========================================================
@torch.no_grad()
def p_sample(x_t, t_scalar):
    """
    One reverse step for all samples in batch at timestep t_scalar (int).
    x_t: (N,2)
    """
    N = x_t.size(0)
    t = torch.full((N,), t_scalar, device=device, dtype=torch.long)  # (N,)
    eps_pred = eps_model(x_t, t)                                     # (N,2)

    # DDPM mean:
    # x_{t-1} = 1/sqrt(alpha_t) * (x_t - (1-alpha_t)/sqrt(1-a_bar_t)*eps_pred) + sigma_t*z
    a_t = alphas[t_scalar - 1]
    ab_t = alpha_bars[t_scalar - 1]

    coef = (1.0 - a_t) / torch.sqrt(1.0 - ab_t)
    mean = (1.0 / torch.sqrt(a_t)) * (x_t - coef * eps_pred)

    if t_scalar > 1:
        z = torch.randn_like(x_t)
        x_prev = mean + sigmas[t_scalar - 1] * z
    else:
        x_prev = mean  # no noise at final step
    return x_prev

@torch.no_grad()
def sample_ddpm(n_samples=2000):
    """
    Full sampling loop: start from x_T ~ N(0,I), run reverse to x_0.
    """
    x = torch.randn(n_samples, x_dim, device=device)  # x_T
    for t in range(T, 0, -1):
        x = p_sample(x, t)
    return x  # approx samples from data

# =========================================================
# 6. Plot helpers -> frames -> GIFs
# =========================================================
def _setup_axes():
    ax = plt.gca()
    ax.set_aspect("equal", "box")
    ax.set_xlim(-1.6, 1.6)
    ax.set_ylim(-1.6, 1.6)
    ax.grid(True)

@torch.no_grad()
def save_convergence_frame(epoch, n_points=2500):
    x_real = sample_real(n_points).cpu().numpy()
    x_fake = sample_ddpm(n_points).cpu().numpy()

    plt.figure(figsize=(4, 4))
    plt.scatter(x_real[:, 0], x_real[:, 1], s=5, alpha=0.2, label="Real")
    plt.scatter(x_fake[:, 0], x_fake[:, 1], s=5, alpha=0.8, label="DDPM Samples")
    _setup_axes()
    plt.title(f"DDPM samples vs real | epoch {epoch+1}")
    plt.xlabel("x1"); plt.ylabel("x2")
    plt.legend()
    plt.tight_layout()
    frame_path = frames_conv_dir / f"epoch_{epoch+1:03d}.png"
    plt.savefig(frame_path, dpi=150)
    plt.close()

@torch.no_grad()
def save_inference_path_gif_frames(
    n_traj=20,
    save_every=5,
):
    """
    Creates frames that show the *trajectory* of a few samples
    as they reverse-diffuse from x_T ~ N(0,I) to x_0.

    We draw:
      - Real circle points (faint)
      - Current points at this reverse step
      - Trajectory lines for each sample (path so far)
    """
    # Real background
    x_real_bg = sample_real(3000).cpu().numpy()

    # Start from standard normal (these are x_T)
    x = torch.randn(n_traj, x_dim, device=device)
    path = [x.clone()]  # store states

    # Run reverse diffusion and store intermediate states
    for t in range(T, 0, -1):
        x = p_sample(x, t)
        if (t % save_every == 0) or (t == 1):
            path.append(x.clone())

    # Now create animation frames from stored path states
    # path[0] is initial noise, path[-1] is final sample
    # We'll progressively show more of the path
    for k in range(len(path)):
        plt.figure(figsize=(4, 4))

        # background real samples
        plt.scatter(x_real_bg[:, 0], x_real_bg[:, 1], s=5, alpha=0.15, label="Real")

        # draw trajectories up to step k
        pts = torch.stack(path[:k+1], dim=0).cpu().numpy()  # (k+1, n_traj, 2)
        for i in range(n_traj):
            plt.plot(pts[:, i, 0], pts[:, i, 1], linewidth=1, alpha=0.8)

        # current points (last in shown path)
        cur = pts[-1]
        plt.scatter(cur[:, 0], cur[:, 1], s=25, alpha=0.9, label="Current")

        _setup_axes()
        plt.title(f"Reverse diffusion path (frame {k+1}/{len(path)})")
        plt.xlabel("x1"); plt.ylabel("x2")
        plt.legend()
        plt.tight_layout()

        frame_path = frames_path_dir / f"path_{k+1:03d}.png"
        plt.savefig(frame_path, dpi=150)
        plt.close()

def make_gif(frames_folder: Path, gif_file: Path, duration=0.20):
    frame_files = sorted(frames_folder.glob("*.png"))
    images = [imageio.imread(f) for f in frame_files]
    imageio.mimsave(gif_file, images, duration=duration, loop=0)

# =========================================================
# 7. Training loop + convergence GIF frames
# =========================================================
for epoch in range(num_epochs):
    eps_model.train()
    running_loss = 0.0

    for _ in range(steps_per_epoch):
        x0 = sample_real(batch_size)  # (B,2)

        # sample random t in [1..T]
        t = torch.randint(1, T + 1, (batch_size,), device=device, dtype=torch.long)

        # forward diffuse
        x_t, eps = q_sample(x0, t)

        # predict eps
        eps_pred = eps_model(x_t, t)

        loss = F.mse_loss(eps_pred, eps)

        opt.zero_grad()
        loss.backward()
        opt.step()

        running_loss += loss.item()

    running_loss /= steps_per_epoch
    print(f"Epoch {epoch+1}/{num_epochs} | eps_mse={running_loss:.6f}")

    # Save convergence frame each epoch
    save_convergence_frame(epoch, n_points=2500)

# Build convergence GIF
make_gif(frames_conv_dir, gif_convergence, duration=0.20)
print(f"Saved convergence GIF: {gif_convergence.resolve()}")

# =========================================================
# 8. Inference-path GIF (post-training)
# =========================================================
# 1) Create frames showing reverse diffusion trajectories
save_inference_path_gif_frames(n_traj=20, save_every=5)

# 2) Build inference-path GIF
make_gif(frames_path_dir, gif_inference_path, duration=0.18)
print(f"Saved inference-path GIF: {gif_inference_path.resolve()}")

Epoch 1/50 | eps_mse=0.568301
Epoch 2/50 | eps_mse=0.489527
Epoch 3/50 | eps_mse=0.486692
Epoch 4/50 | eps_mse=0.486688
Epoch 5/50 | eps_mse=0.485228
Epoch 6/50 | eps_mse=0.480916
Epoch 7/50 | eps_mse=0.482367
Epoch 8/50 | eps_mse=0.479160
Epoch 9/50 | eps_mse=0.480715
Epoch 10/50 | eps_mse=0.477235
Epoch 11/50 | eps_mse=0.472859
Epoch 12/50 | eps_mse=0.455977
Epoch 13/50 | eps_mse=0.422090
Epoch 14/50 | eps_mse=0.402738
Epoch 15/50 | eps_mse=0.395424
Epoch 16/50 | eps_mse=0.386016
Epoch 17/50 | eps_mse=0.381648
Epoch 18/50 | eps_mse=0.381407
Epoch 19/50 | eps_mse=0.380176
Epoch 20/50 | eps_mse=0.378898
Epoch 21/50 | eps_mse=0.379401
Epoch 22/50 | eps_mse=0.374944
Epoch 23/50 | eps_mse=0.375645
Epoch 24/50 | eps_mse=0.373719
Epoch 25/50 | eps_mse=0.372132
Epoch 26/50 | eps_mse=0.374270
Epoch 27/50 | eps_mse=0.370212
Epoch 28/50 | eps_mse=0.374389
Epoch 29/50 | eps_mse=0.373384
Epoch 30/50 | eps_mse=0.372063
Epoch 31/50 | eps_mse=0.370765
Epoch 32/50 | eps_mse=0.370363
Epoch 33/50 | eps

In [8]:
import numpy as np
from pathlib import Path
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import torch

# ----------------------------
# GIF config
# ----------------------------
out_dir = Path("ddpm_medium_gifs")
out_dir.mkdir(parents=True, exist_ok=True)

frames_fwd = out_dir / "frames_forward"
frames_rev = out_dir / "frames_reverse"
frames_fwd.mkdir(exist_ok=True, parents=True)
frames_rev.mkdir(exist_ok=True, parents=True)

gif_forward = out_dir / "ddpm_forward_noising_medium.gif"
gif_reverse = out_dir / "ddpm_reverse_denoising_medium.gif"

# ----------------------------
# Helpers
# ----------------------------
def make_gif(frames_folder: Path, gif_file: Path, duration=0.18):
    frame_files = sorted(frames_folder.glob("*.png"))
    images = [imageio.imread(f) for f in frame_files]
    imageio.mimsave(gif_file, images, duration=duration, loop=0)
    print(f"Saved: {gif_file.resolve()}")

def setup_axes(ax):
    ax.set_aspect("equal", "box")
    ax.set_xlim(-1.6, 1.6)
    ax.set_ylim(-1.6, 1.6)
    ax.grid(True)

# ----------------------------
# 1) Forward NOISING (q): x0 -> x_t
# ----------------------------
@torch.no_grad()
def save_forward_noising_gif(
    n_points=2500,
    n_frames=40,         # "medium": 30-50 looks good
    start_t=1,
    end_t=None,
    fixed_eps=True
):
    if end_t is None:
        end_t = T

    # fixed x0
    x0 = sample_real(n_points)

    # use the SAME eps across timesteps (so the motion looks coherent)
    eps = torch.randn_like(x0) if fixed_eps else None

    # choose timesteps evenly spaced
    ts = np.linspace(start_t, end_t, n_frames).round().astype(int)
    ts = np.clip(ts, 1, T)

    x_real_bg = sample_real(3000).cpu().numpy()

    for i, t_scalar in enumerate(ts, start=1):
        t = torch.full((n_points,), int(t_scalar), device=device, dtype=torch.long)
        x_t, _ = q_sample(x0, t, eps=eps if fixed_eps else None)

        plt.figure(figsize=(4, 4))
        # background: real
        plt.scatter(x_real_bg[:, 0], x_real_bg[:, 1], s=5, alpha=0.12, label="Real (bg)")
        # current noised points
        xt = x_t.cpu().numpy()
        plt.scatter(xt[:, 0], xt[:, 1], s=5, alpha=0.85, label=f"Noised x_t (t={t_scalar})")

        ax = plt.gca()
        setup_axes(ax)
        plt.title(f"Forward diffusion (noising) | frame {i}/{len(ts)}")
        plt.xlabel("x1"); plt.ylabel("x2")
        plt.legend(loc="upper right")
        plt.tight_layout()

        plt.savefig(frames_fwd / f"fwd_{i:03d}.png", dpi=150)
        plt.close()

    make_gif(frames_fwd, gif_forward, duration=0.18)

# ----------------------------
# 2) Reverse DENOISING (pθ): x_T -> x0
# ----------------------------
@torch.no_grad()
def save_reverse_denoising_gif(
    n_points=2500,
    n_frames=40,   # "medium"
    start_t=None,
    end_t=1
):
    if start_t is None:
        start_t = T

    # start from standard normal at x_T
    x = torch.randn(n_points, x_dim, device=device)

    # choose timesteps evenly spaced (descending)
    ts = np.linspace(start_t, end_t, n_frames).round().astype(int)
    ts = np.clip(ts, 1, T)
    ts = list(ts)

    x_real_bg = sample_real(3000).cpu().numpy()

    # We'll "jump" by running the model through the missing steps between saved frames
    # to keep the reverse path correct, while only saving medium number of frames.
    current_t = start_t

    frame_idx = 1
    for target_t in ts:
        # run reverse steps from current_t down to target_t
        for t_scalar in range(int(current_t), int(target_t)-1, -1):
            x = p_sample(x, t_scalar)
        current_t = target_t

        plt.figure(figsize=(4, 4))
        plt.scatter(x_real_bg[:, 0], x_real_bg[:, 1], s=5, alpha=0.12, label="Real (bg)")
        xs = x.cpu().numpy()
        plt.scatter(xs[:, 0], xs[:, 1], s=5, alpha=0.85, label=f"Denoised (t={int(target_t)})")

        ax = plt.gca()
        setup_axes(ax)
        plt.title(f"Reverse diffusion (denoising) | frame {frame_idx}/{len(ts)}")
        plt.xlabel("x1"); plt.ylabel("x2")
        plt.legend(loc="upper right")
        plt.tight_layout()

        plt.savefig(frames_rev / f"rev_{frame_idx:03d}.png", dpi=150)
        plt.close()
        frame_idx += 1

    make_gif(frames_rev, gif_reverse, duration=0.18)

# ----------------------------
# Run both GIF generators
# ----------------------------
save_forward_noising_gif(n_points=2500, n_frames=40, fixed_eps=True)
save_reverse_denoising_gif(n_points=2500, n_frames=40)

Saved: /content/ddpm_medium_gifs/ddpm_forward_noising_medium.gif
Saved: /content/ddpm_medium_gifs/ddpm_reverse_denoising_medium.gif
